# Learn sentiment words from labeled Steam reviews

Load all game CSVs in `data/reviews` using `load_reviews`. Deduplicate by `(appid, recommendationid)` before counting; prefer the last labeled occurrence. Rows with missing labels are reported and excluded, not treated as negative. The balance report describes the labeled reviews actually used.

Use `ReviewParser.process` to tokenize and remove stop words. Count each word **once per review**, regardless of repeated mentions. Here $P(w\mid+)$ means the probability that a positive review contains the word, rather than its share of positive tokens.

With add-one smoothing:

$$P(w\mid+) = \frac{df_+(w)+\alpha}{N_+ + 2\alpha},\qquad P(w\mid-) = \frac{df_-(w)+\alpha}{N_- + 2\alpha}$$

$$S(w)=\log\left(\frac{P(w\mid+)}{P(w\mid-)}\right)$$

Keep words appearing in at least **50 distinct labeled reviews** and with $|S(w)|\geq\log(1.5)$ by default. Thus a word must be at least 1.5 times as prevalent in one class. Both thresholds are configurable. Positive scores associate with positive reviews; negative scores associate with negative reviews. This is association, not a significance test, and does not interpret negation or control for game-specific vocabulary. No reviews are downsampled to balance classes.


In [9]:
import csv
import json
import math
import sys
from collections import Counter
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src").is_dir() and (path / "notebooks").is_dir()), None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start from the project root or notebooks folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.csv_io import REVIEWS_DATA_DIR, load_reviews
from src.review_parser import ReviewParser

MIN_REVIEW_COUNT = 3000
MIN_ABS_SCORE = math.log(1)
ALPHA = 1.0
DATA_DIR = PROJECT_ROOT / "data"


In [2]:
parser = ReviewParser()
positive_counts = Counter()
negative_counts = Counter()
positive_reviews = negative_reviews = unlabeled_reviews = duplicate_rows = 0
files = sorted(
    (path for path in REVIEWS_DATA_DIR.glob("*.csv")
     if path.stem.isascii() and path.stem.isdigit()
     and str(int(path.stem)) == path.stem and 0 < int(path.stem) <= 2**32 - 1),
    key=lambda path: int(path.stem),
)
for index, path in enumerate(files, start=1):
    unique_reviews = {}
    for review in load_reviews(int(path.stem)):
        key = (review.appid, review.recommendationid)
        if key in unique_reviews:
            duplicate_rows += 1
        if key not in unique_reviews or review.voted_up is not None:
            unique_reviews[key] = review
    for review in unique_reviews.values():
        if review.voted_up is None:
            unlabeled_reviews += 1
            continue
        words = {
            token for clause in parser.process(review.review) for token in clause
            if any(character.isalpha() for character in token)
        }
        if review.voted_up:
            positive_reviews += 1
            positive_counts.update(words)
        else:
            negative_reviews += 1
            negative_counts.update(words)
    if index % 100 == 0 or index == len(files):
        print(f"Processed {index}/{len(files)} game files", flush=True)

labeled_reviews = positive_reviews + negative_reviews
balance = {
    "game_files": len(files),
    "labeled_reviews": labeled_reviews,
    "positive_reviews": positive_reviews,
    "negative_reviews": negative_reviews,
    "positive_percent": 100 * positive_reviews / labeled_reviews if labeled_reviews else 0,
    "negative_percent": 100 * negative_reviews / labeled_reviews if labeled_reviews else 0,
    "majority_to_minority_ratio": (
        max(positive_reviews, negative_reviews) / min(positive_reviews, negative_reviews)
        if positive_reviews and negative_reviews else None
    ),
    "unlabeled_reviews_excluded": unlabeled_reviews,
    "duplicate_rows_excluded": duplicate_rows,
}
print(json.dumps(balance, indent=2))


Processed 100/663 game files
Processed 200/663 game files
Processed 300/663 game files
Processed 400/663 game files
Processed 500/663 game files
Processed 600/663 game files
Processed 663/663 game files
{
  "game_files": 663,
  "labeled_reviews": 63858,
  "positive_reviews": 50221,
  "negative_reviews": 13637,
  "positive_percent": 78.64480566256381,
  "negative_percent": 21.355194337436185,
  "majority_to_minority_ratio": 3.6827014739312167,
  "unlabeled_reviews_excluded": 0,
  "duplicate_rows_excluded": 0
}


In [10]:
def calculate_word_scores(positive_counts, negative_counts, n_positive, n_negative,
                          min_reviews=50, min_abs_score=math.log(1.5), alpha=1.0):
    """Calculate smoothed review-presence log ratios and filter weak/rare words."""
    if n_positive <= 0 or n_negative <= 0:
        raise ValueError("Both positive and negative labeled reviews are required")
    if alpha <= 0 or min_reviews < 1 or min_abs_score < 0:
        raise ValueError("Require alpha > 0, min_reviews >= 1, min_abs_score >= 0")
    scores = []
    for word in positive_counts.keys() | negative_counts.keys():
        positive = positive_counts[word]
        negative = negative_counts[word]
        review_count = positive + negative
        if review_count < min_reviews:
            continue
        p_positive = (positive + alpha) / (n_positive + 2 * alpha)
        p_negative = (negative + alpha) / (n_negative + 2 * alpha)
        score = math.log(p_positive / p_negative)
        if abs(score) < min_abs_score:
            continue
        scores.append({
            "word": word, "score": score,
            "association": "positive" if score > 0 else "negative" if score < 0 else "neutral",
            "review_count": review_count,
            "positive_review_count": positive, "negative_review_count": negative,
            "p_positive": p_positive, "p_negative": p_negative,
        })
    return sorted(scores, key=lambda row: (-abs(row["score"]), row["word"]))

word_scores = calculate_word_scores(
    positive_counts, negative_counts, positive_reviews, negative_reviews,
    MIN_REVIEW_COUNT, MIN_ABS_SCORE, ALPHA,
)
print(f"Retained {len(word_scores)} words")
for association in ("positive", "negative"):
    print(f"\nStrongest {association} associations:")
    for row in [row for row in word_scores if row["association"] == association][:20]:
        print(f"{row['word']:24s} S={row['score']:+.3f} reviews={row['review_count']}")


Retained 312 words

Strongest positive associations:
master                   S=+1.860 reviews=3081
souls                    S=+1.729 reviews=3109
dark                     S=+1.365 reviews=4437
beautiful                S=+1.285 reviews=5638
minor                    S=+1.214 reviews=3146
learn                    S=+1.209 reviews=4692
highly                   S=+1.173 reviews=3081
recommended              S=+1.160 reviews=3707
average                  S=+1.067 reviews=3122
u                        S=+1.019 reviews=3213
definitely               S=+1.001 reviews=3920
computer                 S=+0.986 reviews=3184
survival                 S=+0.975 reviews=3645
easy                     S=+0.898 reviews=7023
difficulty               S=+0.876 reviews=5275
music                    S=+0.873 reviews=3905
perfect                  S=+0.868 reviews=3525
difficult                S=+0.866 reviews=4367
graphics                 S=+0.837 reviews=8687
check                    S=+0.826 reviews=3281

Strong

In [11]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
SCORES_FILE = DATA_DIR / "word_sentiment_scores.csv"
REPORT_FILE = DATA_DIR / "word_sentiment_report.json"
fields = ["word", "score", "association", "review_count", "positive_review_count",
          "negative_review_count", "p_positive", "p_negative"]
with SCORES_FILE.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fields)
    writer.writeheader()
    writer.writerows(word_scores)
report = {**balance, "min_review_count": MIN_REVIEW_COUNT,
          "min_abs_score": MIN_ABS_SCORE, "smoothing_alpha": ALPHA,
          "retained_words": len(word_scores)}
REPORT_FILE.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(f"Saved {SCORES_FILE} and {REPORT_FILE}")


Saved C:\projects\GameRecommender\data\word_sentiment_scores.csv and C:\projects\GameRecommender\data\word_sentiment_report.json
